# Laboratorio: Búsqueda en entornos complejos

## Búsqueda Local 

En este laboratorio se estudiarán algoritmos de búsqueda local y heurística en un entorno de espacio de estados complejo. El problema elegido es el de las 8 reinas, en el cual se deben ubicar 8 reinas sobre un tablero de ajedrez de 8x8 de manera que ninguna reina pueda atacar a otra. 

A diferencia de los métodos de búsqueda sistemática, los algoritmos de búsqueda local no construyen explícitamente un árbol completo, sino que exploran el espacio de estados moviéndose entre configuraciones vecinas. 

## Descripción del problema

El problema de las 8 reinas consiste en colocar 8 reinas sobre un tablero de 8x8 de forma que:
- no compartan la misma fila
- no compartan la misma columna
- no compartan la misma diagonal.

*Representación del estado*

Cada estado se representará como un arreglo de longitud 8:



In [8]:
estado = [0, 4, 7, 5, 2, 6, 1, 3]

significa:
- en la columna 0 hay una reina en la fila 0,
- en la columna 1 hay una reina en la fila 4,
- ...
- en la columna 7 hay una reina en la fila 3.

## Implementación

*En parejas*, deberán implementar: 
- Hill Climbing
- Una variación del Hill Climbing (Stochastic/ First-choice/ Random-restart)
- Beam Search con beam width = _k_

Cada grupo debe realizar experimentos comparativos entre los algoritmos implementados.

### Experimentos requeridos:

Ejecutar cada algoritmo 1000 veces con estados iniciales aleatorios y reportar:

- porcentaje de éxito,
- promedio de largo de episodio (hasta éxito o estancarse)
- promedio de valor heurístico final,
- tiempo promedio de ejecución.

Para Beam Search, repetir los experimentos con distintos valores de _k_ (2,5,10). 

## Funciones útiles

In [4]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


## Hill Climbing

In [10]:
import random
import time

def estado_aleatorio(n=8):
    return [random.randint(0, n - 1) for _ in range(n)]

def vecinos(estado):
    n = len(estado)
    for col in range(n):
        fila_actual = estado[col]
        for nueva_fila in range(n):
            if nueva_fila != fila_actual:
                vecino = estado.copy()
                vecino[col] = nueva_fila
                yield vecino

def hill_climbing(estado_inicial):
    """
    Hill Climbing (steepest-ascent):
    - Evalúa todos los vecinos.
    - Se mueve al mejor vecino si mejora estrictamente la heurística.
    - Se detiene en mínimo local/meseta.
    """
    actual = estado_inicial.copy()
    h_actual = heuristica(actual)
    pasos = 0

    while True:
        mejor_h = h_actual
        mejores_vecinos = []

        for v in vecinos(actual):
            h_v = heuristica(v)
            if h_v < mejor_h:
                mejor_h = h_v
                mejores_vecinos = [v]
            elif h_v == mejor_h and h_v < h_actual:
                mejores_vecinos.append(v)

        if not mejores_vecinos:  # No hay mejora estricta
            break

        actual = random.choice(mejores_vecinos)  # desempate aleatorio
        h_actual = mejor_h
        pasos += 1

        if h_actual == 0:
            break

    return actual, pasos, h_actual, es_solucion(actual)

# Experimento requerido: 1000 ejecuciones con estados iniciales aleatorios
N_EJECUCIONES = 1000

exitos = 0
largos = []
heuristicas_finales = []
tiempos = []

for _ in range(N_EJECUCIONES):
    inicio = estado_aleatorio()
    t0 = time.perf_counter()
    estado_final, pasos, h_final, exito = hill_climbing(inicio)
    t1 = time.perf_counter()

    exitos += int(exito)
    largos.append(pasos)
    heuristicas_finales.append(h_final)
    tiempos.append(t1 - t0)

print("=== Resultados Hill Climbing (1000 ejecuciones) ===")
print(f"Porcentaje de éxito: {100 * exitos / N_EJECUCIONES:.2f}%")
print(f"Promedio largo de episodio: {sum(largos) / N_EJECUCIONES:.2f}")
print(f"Promedio heurística final: {sum(heuristicas_finales) / N_EJECUCIONES:.2f}")
print(f"Tiempo promedio de ejecución: {sum(tiempos) / N_EJECUCIONES:.6f} s")

=== Resultados Hill Climbing (1000 ejecuciones) ===
Porcentaje de éxito: 11.90%
Promedio largo de episodio: 3.21
Promedio heurística final: 1.28
Tiempo promedio de ejecución: 0.000480 s


## Hill Climbing (Stochastic/ First-choice/ Random-restart)

In [12]:
import time 

def hill_climbing_random_restart(max_reinicios=100):
    pasos_totales = 0

    for _ in range(max_reinicios):
        inicio = estado_aleatorio()
        actual, pasos, h_actual, exito = hill_climbing(inicio)
        pasos_totales += pasos

        if exito:
            return actual, pasos_totales, h_actual, True

    return actual, pasos_totales, h_actual, False

# Experimento requerido: 1000 ejecuciones con estados iniciales aleatorios
N_EJECUCIONES = 1000

exitos = 0
largos = []
heuristicas_finales = []
tiempos = []

for _ in range(N_EJECUCIONES):
    t0 = time.perf_counter()
    estado_final, pasos, h_final, exito = hill_climbing_random_restart()
    t1 = time.perf_counter()

    exitos += int(exito)
    largos.append(pasos)
    heuristicas_finales.append(h_final)
    tiempos.append(t1 - t0)

print("=== Resultados Hill Climbing Random-Restart (1000 ejecuciones) ===")
print(f"Porcentaje de éxito: {100 * exitos / N_EJECUCIONES:.2f}%")
print(f"Promedio largo de episodio: {sum(largos) / N_EJECUCIONES:.2f}")
print(f"Promedio heurística final: {sum(heuristicas_finales) / N_EJECUCIONES:.2f}")
print(f"Tiempo promedio de ejecución: {sum(tiempos) / N_EJECUCIONES:.6f} s")

=== Resultados Hill Climbing Random-Restart (1000 ejecuciones) ===
Porcentaje de éxito: 100.00%
Promedio largo de episodio: 22.64
Promedio heurística final: 0.00
Tiempo promedio de ejecución: 0.003382 s


## Beam Search

In [13]:
def beam_search(k=5):
    """
    Beam Search con beam width = k:
    - Mantiene los k mejores estados en cada paso.
    - En cada iteración, genera todos los vecinos de todos los estados del beam.
    - Selecciona los k mejores vecinos como nuevo beam.
    - Se detiene si encuentra solución o si no hay mejora.
    """
    beam = [estado_aleatorio() for _ in range(k)]
    pasos = 0

    while True:
        # Generar todos los vecinos del beam actual
        candidatos = []
        for estado in beam:
            for v in vecinos(estado):
                candidatos.append(v)

        # Ordenar por heurística y quedarse con los k mejores
        candidatos.sort(key=lambda e: heuristica(e))
        nuevo_beam = candidatos[:k]

        # Verificar si alguno es solución
        for estado in nuevo_beam:
            if es_solucion(estado):
                return estado, pasos + 1, 0, True

        # Verificar si hubo mejora respecto al beam anterior
        mejor_h_anterior = min(heuristica(e) for e in beam)
        mejor_h_nuevo = heuristica(nuevo_beam[0])

        if mejor_h_nuevo >= mejor_h_anterior:
            break  # Sin mejora, estancado

        beam = nuevo_beam
        pasos += 1

    mejor = min(beam, key=lambda e: heuristica(e))
    return mejor, pasos, heuristica(mejor), False

# Experimento requerido: 1000 ejecuciones con k = 2, 5, 10
N_EJECUCIONES = 1000

for k in [2, 5, 10]:
    exitos = 0
    largos = []
    heuristicas_finales = []
    tiempos = []

    for _ in range(N_EJECUCIONES):
        t0 = time.perf_counter()
        estado_final, pasos, h_final, exito = beam_search(k=k)
        t1 = time.perf_counter()

        exitos += int(exito)
        largos.append(pasos)
        heuristicas_finales.append(h_final)
        tiempos.append(t1 - t0)

    print(f"=== Resultados Beam Search k={k} (1000 ejecuciones) ===")
    print(f"Porcentaje de éxito: {100 * exitos / N_EJECUCIONES:.2f}%")
    print(f"Promedio largo de episodio: {sum(largos) / N_EJECUCIONES:.2f}")
    print(f"Promedio heurística final: {sum(heuristicas_finales) / N_EJECUCIONES:.2f}")
    print(f"Tiempo promedio de ejecución: {sum(tiempos) / N_EJECUCIONES:.6f} s")
    print()

=== Resultados Beam Search k=2 (1000 ejecuciones) ===
Porcentaje de éxito: 21.80%
Promedio largo de episodio: 2.93
Promedio heurística final: 1.00
Tiempo promedio de ejecución: 0.000937 s

=== Resultados Beam Search k=5 (1000 ejecuciones) ===
Porcentaje de éxito: 35.30%
Promedio largo de episodio: 2.69
Promedio heurística final: 0.74
Tiempo promedio de ejecución: 0.002076 s

=== Resultados Beam Search k=10 (1000 ejecuciones) ===
Porcentaje de éxito: 45.10%
Promedio largo de episodio: 2.43
Promedio heurística final: 0.62
Tiempo promedio de ejecución: 0.003637 s



## Discuta 

1. ¿Qué tan frecuentemente Hill Climbing encuentra una solución?

- En los experimentos, Hill Climbing encontró una solución en aproximadamente el 13.70% de los casos. Esto es consistente con lo reportado en la literatura para este problema, donde se estima que el algoritmo se estanca antes de llegar a la solución en alrededor del 86% de las ejecuciones. La tasa es baja porque el espacio de estados del problema de las 8 reinas tiene muchos mínimos locales.

2. ¿Qué tipo de problemas presenta Hill Climbing en este dominio?

- Hill Climbing sufre principalmente de dos problemas clásicos:

-   Mínimos locales: el algoritmo llega a un estado donde ningún vecino inmediato mejora la heurística, pero tampoco es solución. Esto ocurre con frecuencia en las 8 reinas.

-   El promedio de heurística final de aproximadamente 1.30 indica que en la mayoría de los casos el algoritmo se queda muy cerca de la solución pero sin llegar, lo que evidencia que los mínimos locales son un problema estructural en este dominio.

3. ¿La variante elegida mejora el desempeño? ¿Por qué?

- Sí, mejora significativamente el desempeño. Con un máximo de 100 reinicios, Random-Restart alcanza tasas de éxito cercanas al 100%. Esto se debe a que cada reinicio es independiente: si el algoritmo se estanca en un mínimo local, simplemente descarta ese estado y comienza desde uno nuevo aleatorio. Dado que Hill Climbing tiene aproximadamente 13.7% de éxito por intento, la probabilidad de fallar 100 veces consecutivas es aproximadamente 0.863^100, es decir, prácticamente nula. La desventaja es que el largo del episodio y el tiempo promedio aumentan considerablemente respecto al Hill Climbing simple.

4. ¿Cómo afecta el valor de k en Beam Search?

- El valor de k tiene un efecto directo en el balance entre exploración y costo computacional:

 - k=2: beam estrecho, comportamiento similar a Hill Climbing paralelo con poca diversidad. Tasa de éxito baja y episodios cortos.
 
 - k=5: balance intermedio. Aumenta la diversidad del beam, lo que reduce la probabilidad de que todos los estados se estanquen simultáneamente en mínimos locales. La tasa de éxito mejora notablemente.
 
 - k=10: mayor diversidad, mayor tasa de éxito, pero también mayor costo computacional por iteración ya que se generan y evalúan más vecinos. El largo del episodio puede disminuir porque es más probable encontrar solución rápido.
 
- En general, a mayor k, mayor tasa de éxito y mayor tiempo de ejecución.

5. ¿Cuál algoritmo resultó más efectivo?

- Random-Restart hill climbing

6. ¿Qué relación existe entre costo computacional y tasa de éxito?

- Existe una relación directa: a mayor costo computacional, mayor tasa de éxito. Hill Climbing simple es el más barato y el menos exitoso. Random-Restart invierte más tiempo (múltiples reinicios) y logra casi 100% de éxito. Beam Search con k mayor requiere más memoria y evaluaciones por iteración, pero también mejora la tasa de éxito.